# Spark + Hive + MinIO smoke test

Writes a managed Delta table through the Hive metastore, backed by MinIO (`s3a://`), then reads it back and verifies the table's location. Requires `hive-metastore` and `minio` running (`docker compose up -d`).

In [1]:
%load_ext sparksql_magic

In [2]:
from pyspark.sql import SparkSession

DB_NAME = "test_db"
TABLE_NAME = "hive_minio_check"
WAREHOUSE_DIR = "s3a://data-platform/managed"

spark = (
    SparkSession.builder
    .appName("hive-minio-test")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "spark-master")
    .config("spark.sql.warehouse.dir", WAREHOUSE_DIR)
    .enableHiveSupport()
    .getOrCreate()
)

26/08/15 13:19:07 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
%%sparksql
SELECT * FROM test_db.hive_minio_check ORDER BY id

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/08/15 13:19:18 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

id,name,age
1,Alice,30
2,Bob,30
3,Carol,30


In [3]:
spark.sql("SHOW DATABASES").show(truncate=False)

+---------+
|namespace|
+---------+
|default  |
|test_db  |
+---------+



In [2]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME} LOCATION '{WAREHOUSE_DIR}/{DB_NAME}.db'")
spark.sql("SHOW DATABASES").show(truncate=False)

+---------+
|namespace|
+---------+
|default  |
|test_db  |
+---------+



In [4]:
from pyspark import pipelines as dp

In [3]:
spark.sql(f"DROP TABLE IF EXISTS {DB_NAME}.{TABLE_NAME}")

df = spark.createDataFrame(
    [(1, "Alice", 30), (2, "Bob", 25), (3, "Carol", 35)],
    ["id", "name", "age"],
)
df.write.format("delta").mode("overwrite").saveAsTable(f"{DB_NAME}.{TABLE_NAME}")
print(f"Wrote table {DB_NAME}.{TABLE_NAME}")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/08/15 10:46:16 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`test_db`.`hive_minio_check` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
26/08/15 10:46:16 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


Wrote table test_db.hive_minio_check


In [4]:
df.show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1|Alice| 30|
|  2|  Bob| 25|
|  3|Carol| 35|
+---+-----+---+



In [6]:
spark.sql("""
update test_db.hive_minio_check
set age = 26
where id = 1
""")

Py4JJavaError: An error occurred while calling o39.sql.
: org.apache.spark.SparkException: [INTERNAL_ERROR] No active or default Spark session found SQLSTATE: XX000
	at org.apache.spark.SparkException$.internalError(SparkException.scala:92)
	at org.apache.spark.SparkException$.internalError(SparkException.scala:96)
	at org.apache.spark.sql.SparkSessionCompanion.$anonfun$active$2(SparkSession.scala:1031)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.sql.SparkSessionCompanion.$anonfun$active$1(SparkSession.scala:1031)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.sql.SparkSessionCompanion.active(SparkSession.scala:1030)
	at org.apache.spark.sql.delta.sources.DeltaDataSource.getTable(DeltaDataSource.scala:112)
	at org.apache.spark.sql.execution.datasources.v2.V2SessionCatalog.loadTable(V2SessionCatalog.scala:119)
	at org.apache.spark.sql.connector.catalog.TableCatalog.loadTable(TableCatalog.java:169)
	at org.apache.spark.sql.execution.datasources.v2

In [5]:
# Read back through the Hive catalog (proves the table is registered in Hive)
spark.sql(f"SHOW TABLES IN {DB_NAME}").show(truncate=False)
spark.sql(f"SELECT * FROM {DB_NAME}.{TABLE_NAME} ORDER BY id").show()

+---------+----------------+-----------+
|namespace|tableName       |isTemporary|
+---------+----------------+-----------+
|test_db  |external_test   |false      |
|test_db  |hive_minio_check|false      |
+---------+----------------+-----------+



+---+-----+---+
| id| name|age|
+---+-----+---+
|  1|Alice| 26|
|  2|  Bob| 25|
|  3|Carol| 35|
+---+-----+---+



26/08/15 12:41:15 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: Master removed our application: KILLED
26/08/15 12:41:15 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exiting due to error from cluster scheduler: Master removed our application: KILLED
	at org.apache.spark.errors.SparkCoreErrors$.clusterSchedulerError(SparkCoreErrors.scala:282)
	at org.apache.spark.scheduler.TaskSchedulerImpl.error(TaskSchedulerImpl.scala:922)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend.dead(StandaloneSchedulerBackend.scala:166)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint.markDead(StandaloneAppClient.scala:273)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint$$anonfun$receive$1.applyOrElse(StandaloneAppClient.scala:174)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:116)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:216)
	at org.apache.spark.rpc.netty.Inbox.proce

In [13]:
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 34.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 31.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.3/508.3 KB 39.3 MB/s eta 0:00:00
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
import pandas as pd

pdf = spark.sql(
    f"DESCRIBE HISTORY {DB_NAME}.{TABLE_NAME}"
).toPandas()

pdf

26/08/15 10:12:23 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
                                                                                

,version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,1,2026-08-15 10:08:41,None,None,UPDATE,"{'predicate': '[""(id#287L = 1)""]'}",None,None,None,0.0,Serializable,False,"{'numDeletionVectorsUpdated': '0', 'numAddedFi...",None,Apache-Spark/4.0.1 Delta-Lake/4.3.1
1,0,2026-08-15 10:07:02,None,None,CREATE OR REPLACE TABLE AS SELECT,"{'partitionBy': '[]', 'description': None, 'pr...",None,None,None,NaN,Serializable,False,"{'numOutputRows': '3', 'numRemovedBytes': '0',...",None,Apache-Spark/4.0.1 Delta-Lake/4.3.1


In [3]:
spark.sql(f"""
RESTORE TABLE {DB_NAME}.{TABLE_NAME}
TO VERSION AS OF 0
""")

26/08/15 10:12:55 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

DataFrame[table_size_after_restore: bigint, num_of_files_after_restore: bigint, num_removed_files: bigint, num_restored_files: bigint, removed_files_size: bigint, restored_files_size: bigint]

In [4]:
spark.sql(f"SELECT * FROM {DB_NAME}.{TABLE_NAME} ORDER BY id").show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1|Alice| 30|
|  2|  Bob| 25|
|  3|Carol| 35|
+---+-----+---+



In [5]:
import pandas as pd

pdf = spark.sql(
    f"DESCRIBE HISTORY {DB_NAME}.{TABLE_NAME}"
).toPandas()

pdf

,version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2,2026-08-15 10:13:04,None,None,RESTORE,"{'version': '0', 'timestamp': None}",None,None,None,1.0,Serializable,False,"{'removedFilesSize': '1017', 'tableSizeAfterRe...",None,Apache-Spark/4.0.1 Delta-Lake/4.3.1
1,1,2026-08-15 10:08:41,None,None,UPDATE,"{'predicate': '[""(id#287L = 1)""]'}",None,None,None,0.0,Serializable,False,"{'numDeletionVectorsUpdated': '0', 'numAddedFi...",None,Apache-Spark/4.0.1 Delta-Lake/4.3.1
2,0,2026-08-15 10:07:02,None,None,CREATE OR REPLACE TABLE AS SELECT,"{'partitionBy': '[]', 'description': None, 'pr...",None,None,None,NaN,Serializable,False,"{'numOutputRows': '3', 'numRemovedBytes': '0',...",None,Apache-Spark/4.0.1 Delta-Lake/4.3.1


In [5]:
# Confirm the table's data actually lives in MinIO (s3a://), not local disk
location = (
    spark.sql(f"DESCRIBE EXTENDED {DB_NAME}.{TABLE_NAME}")
    .filter("col_name = 'Location'")
    .collect()[0]["data_type"]
)
print(f"Location: {location}")
assert location.startswith("s3a://data-platform/"), f"Unexpected table location: {location}"
print("OK: table is registered in Hive metastore and data lives in MinIO (s3a).")

Location: s3a://data-platform/managed/test_db.db/hive_minio_check
OK: table is registered in Hive metastore and data lives in MinIO (s3a).


In [6]:
spark.sql(f"DESCRIBE EXTENDED {DB_NAME}.{TABLE_NAME}").show()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|                  id|              bigint|   NULL|
|                name|              string|   NULL|
|                 age|              bigint|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|                Name|spark_catalog.tes...|       |
|                Type|            EXTERNAL|       |
|            Location|s3a://data-platfo...|       |
|            Provider|               delta|       |
|               Owner|              NBuser|       |
|    Table Properties|[delta.minReaderV...|       |
+--------------------+--------------------+-------+



In [7]:
spark.sql(f"drop table {DB_NAME}.{TABLE_NAME}")

DataFrame[]

In [ ]:
%%sparksql
external_test

In [6]:
TABLE_NAME = "external_test"
LOCATION = "s3a://data-platform/landing/external_test"

# Create external Delta table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DB_NAME}.{TABLE_NAME} (
    id BIGINT,
    name STRING,
    created_at TIMESTAMP
)
USING DELTA
LOCATION '{LOCATION}'
""")

26/08/15 10:15:23 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`test_db`.`external_test` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
26/08/15 10:15:23 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


DataFrame[]

In [7]:
spark.sql(f"""
INSERT INTO {DB_NAME}.{TABLE_NAME} (id, name, created_at)
VALUES
    (1, 'Alice', current_timestamp()),
    (2, 'Bob', current_timestamp()),
    (3, 'Charlie', current_timestamp())
""")

DataFrame[]

In [8]:
df = spark.table(f"{DB_NAME}.{TABLE_NAME}")
df.show()

+---+-------+--------------------+
| id|   name|          created_at|
+---+-------+--------------------+
|  2|    Bob|2026-08-15 10:15:...|
|  3|Charlie|2026-08-15 10:15:...|
|  1|  Alice|2026-08-15 10:15:...|
+---+-------+--------------------+



In [10]:
spark.sql(f"desc formatted {DB_NAME}.{TABLE_NAME}").show()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|                  id|              bigint|   NULL|
|                name|              string|   NULL|
|          created_at|           timestamp|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|                Name|spark_catalog.tes...|       |
|                Type|            EXTERNAL|       |
|            Location|s3a://data-platfo...|       |
|            Provider|               delta|       |
|               Owner|              NBuser|       |
|    Table Properties|[delta.minReaderV...|       |
+--------------------+--------------------+-------+



In [26]:
spark.sql(f"""
DESCRIBE HISTORY {DB_NAME}.{TABLE_NAME}
""").toPandas()

,version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,4,2026-08-15 10:19:33,None,None,OPTIMIZE,"{'predicate': '[]', 'auto': 'false', 'zOrderBy...",None,None,None,3.0,SnapshotIsolation,False,"{'numRemovedBytes': '2109', 'p50FileSize': '10...",None,Apache-Spark/4.0.1 Delta-Lake/4.3.1
1,3,2026-08-15 10:18:59,None,None,RESTORE,"{'version': '1', 'timestamp': None}",None,None,None,2.0,Serializable,False,"{'removedFilesSize': '0', 'tableSizeAfterResto...",None,Apache-Spark/4.0.1 Delta-Lake/4.3.1
2,2,2026-08-15 10:18:42,None,None,RESTORE,"{'version': '0', 'timestamp': None}",None,None,None,1.0,Serializable,False,"{'removedFilesSize': '2109', 'tableSizeAfterRe...",None,Apache-Spark/4.0.1 Delta-Lake/4.3.1
3,1,2026-08-15 10:15:32,None,None,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}",None,None,None,0.0,Serializable,True,"{'numOutputRows': '3', 'numOutputBytes': '2109...",None,Apache-Spark/4.0.1 Delta-Lake/4.3.1
4,0,2026-08-15 10:15:22,None,None,CREATE TABLE,"{'partitionBy': '[]', 'description': None, 'pr...",None,None,None,NaN,Serializable,True,{},None,Apache-Spark/4.0.1 Delta-Lake/4.3.1


In [14]:
spark.sql(f"""
drop table {DB_NAME}.{TABLE_NAME}
""").show()

++
||
++
++



In [15]:
# Create external Delta table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {DB_NAME}.{TABLE_NAME} (
    id BIGINT,
    name STRING,
    created_at TIMESTAMP
)
USING DELTA
LOCATION '{LOCATION}'
""")

26/08/15 10:17:21 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`test_db`.`external_test` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.


DataFrame[]

In [24]:
df = spark.table(f"{DB_NAME}.{TABLE_NAME}")
df.show()

+---+-------+--------------------+
| id|   name|          created_at|
+---+-------+--------------------+
|  2|    Bob|2026-08-15 10:15:...|
|  3|Charlie|2026-08-15 10:15:...|
|  1|  Alice|2026-08-15 10:15:...|
+---+-------+--------------------+



In [23]:
spark.sql(f"""
RESTORE TABLE {DB_NAME}.{TABLE_NAME}
TO VERSION AS OF 1
""")

DataFrame[table_size_after_restore: bigint, num_of_files_after_restore: bigint, num_removed_files: bigint, num_restored_files: bigint, removed_files_size: bigint, restored_files_size: bigint]

In [25]:
spark.sql(f"""
OPTIMIZE {DB_NAME}.{TABLE_NAME}
""")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint>,numBins:bigint,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,

In [4]:
%%sparksql
CREATE TABLE IF NOT EXISTS test_db.test (
    id BIGINT,
    name STRING,
    created_at TIMESTAMP
)
USING DELTA
LOCATION 's3a://data-platform/landing/test'

26/08/15 13:22:37 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`test_db`.`test` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
26/08/15 13:22:37 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


In [5]:
%%sparksql
INSERT INTO test_db.test (id, name, created_at)
VALUES
    (1, 'Alice', current_timestamp()),
    (2, 'Bob', current_timestamp()),
    (3, 'Charlie', current_timestamp())

In [6]:
%%sparksql

select * from test_db.test 

id,name,created_at
2,Bob,2026-08-15 13:23:06.012412
3,Charlie,2026-08-15 13:23:06.012412
1,Alice,2026-08-15 13:23:06.012412
